# 02. Circle houses

**Tier:** Deep. **Why you'd use this:** Houses sit in a circle, so the first and last are neighbors. You still cannot rob two adjacent houses, and you want the largest total.

**After this lesson you can:** reduce a circle to two line problems; reuse the skip-or-take recurrence on each line.
About 25 minutes. **Uses:** [Min cost](01-min-cost-lesson.ipynb), [Memoization](../../course/09-recursion/04-memoization-lesson.ipynb)


## Quick review

1. In a line of houses, what two choices do you compare at each house?
2. Why does a circle make "rob the first and the last" illegal?

<details><summary>Answers</summary>

Skip this house and keep the best so far, or take it and add it to the best from two houses back. The first and last share a wall, so they are adjacent.
</details>


## Recognize it

Values sit in a circle or wrap so the first and last interact. The prompt still forbids adjacent picks and asks for a maximum total.

Signals: "circular", "first and last are adjacent", plus the usual house-robber skip-or-take choice. A look-alike that is not this pattern: a straight line of houses with no wrap — that is the same recurrence once, not the better of two lines.


## Why it works this way

On a line, let `take` be the best total ending after the houses seen so far when the previous house may have been robbed, and let `skip` track the best when you move on without forcing that detail into the next step. The usual roll is: at value `v`, the new best is `max(old_take, old_skip + v)`, and the old best becomes the new skip. Empty is `0`. One house is its value. That recurrence solves the line in one pass with two numbers.

A circle adds one hard edge: house `0` and house `n - 1` are neighbors. Any plan that robs both is illegal. Every legal plan therefore falls into one of two groups: plans that never touch the last house, or plans that never touch the first house. Compute the line answer on `nums[:-1]` and on `nums[1:]`, then take the larger. You do not need a third table for "neither end"; those plans are already included inside both line answers.

Asking the full circle with a naive recursion that tries every subset of non-adjacent houses explodes. Each house branches into take or skip with an adjacency check, and overlapping prefixes repeat. The two line passes each run in time proportional to `n` and constant extra memory, so the circle costs the same order as one line, just twice.

The design reason for the split is the wrap constraint. Encoding "did I rob house 0?" as an extra dimension in one table works, but the two-call split makes the invariant obvious: each call is ordinary line robbery on a range that cannot wrap. See [Min cost](01-min-cost-lesson.ipynb) for the same idea of building an answer from two prior answers, and [Memoization](../../course/09-recursion/04-memoization-lesson.ipynb) when you prefer a top-down `best(i, robbed_first)` form.

Greedy "always take the larger of the next two" fails on the same inputs as linear house robbery. Dynamic programming keeps the better whole path, not the better neighbor.

Marker for uniqueness: 13-dynamic-programming:02-circle-lesson.


## When to use it

Use this when a maximum (or minimum) over choices wraps so the first and last interact, and each position still depends on a short local history such as "took the previous or not". Prefer a single line DP when there is no wrap. Prefer greedy only when a local choice is provably safe for every input, which this adjacency-maximum problem is not. Prefer backtracking when you must list every valid robbery plan instead of one best total.


## Brute force, and why it's slow

Count the subsets. Try every mask, reject adjacent bits and the wrap, and keep the best sum.


In [1]:
def circle_slow(nums):
    n = len(nums)
    best = 0
    checked = 0
    for mask in range(1 << n):
        checked += 1
        ok = True
        total = 0
        for i in range(n):
            if mask & (1 << i):
                if mask & (1 << ((i + 1) % n)):
                    ok = False
                    break
                total += nums[i]
        if ok:
            best = max(best, total)
    return best, checked


for n in (8, 12, 16):
    print(n, circle_slow([2] * n))


8 (8, 256)
12 (12, 4096)
16 (16, 65536)


## Key insight

A legal circle plan never robs both ends. Solve the line on everything but the last house, solve the line on everything but the first, and return the better total.


## Worked example

Houses in a circle hold non-negative values. Return the maximum you can rob without taking two neighbors. The first and last houses are neighbors.


In [2]:
def circle_loot(nums: list[int]) -> int:
    def line(houses: list[int]) -> int:
        # 1. Best total on a straight street.
        skip = take = 0
        for value in houses:
            skip, take = take, max(take, skip + value)
        return take

    # 2. Empty or one house: no wrap conflict.
    if not nums:
        return 0
    if len(nums) == 1:
        return nums[0]
    # 3. Better of dropping the last house or dropping the first.
    return max(line(nums[:-1]), line(nums[1:]))


print(circle_loot([2, 3, 2]))
print(circle_loot([1, 2, 3, 1]))
print(circle_loot([1, 2, 3]))


3
4
3


## Trace

`[2, 3, 2]`:

| range | line steps (skip, take) after each house | line best |
|---|---|---|
| drop last → `[2, 3]` | `(0,2)` then `(2,3)` | **3** |
| drop first → `[3, 2]` | `(0,3)` then `(3,3)` | **3** |

Circle answer `max(3, 3) = 3`. Taking both `2`s would wrap and is illegal.


## Cost

Two linear passes: time grows with the number of houses, and extra memory is a handful of integers. That beats checking every mask, which grows like `2 ** n`.


## Variants

Not solved here: [House Robber](../../challenges/problems/18-house-robber.ipynb) (the line version), [Climbing Stairs](../../challenges/problems/17-climbing-stairs.ipynb), [Coin Change](../../challenges/problems/19-coin-change.ipynb).


## Common mistake

This bug is silent. The function runs the line recurrence on the whole circle and never blocks the wrap. On `[2, 3, 2]` it returns `4` by taking both ends. There is no traceback. The answer is `3`.


In [3]:
def circle_loot_wrong(nums):
    skip = take = 0
    for value in nums:  # treats the circle as a line
        skip, take = take, max(take, skip + value)
    return take


print(circle_loot_wrong([2, 3, 2]))


4


The fix splits into two lines that each drop one end. The buggy cell prints `4` for `[2, 3, 2]` because it treats the circle like an open line and takes both ends.


In [4]:
print(circle_loot([2, 3, 2]))


3


## Python idioms

Tuple assignment rolls `skip` and `take` in one line. A tiny helper keeps the line recurrence in one place so the circle is only `max(line(...), line(...))`.


In [5]:
def circle_loot_named(nums: list[int]) -> int:
    def best_line(houses: list[int]) -> int:
        prev = curr = 0
        for value in houses:
            prev, curr = curr, max(curr, prev + value)
        return curr

    if len(nums) <= 1:
        return nums[0] if nums else 0
    return max(best_line(nums[:-1]), best_line(nums[1:]))


print(circle_loot_named([1, 2, 3, 1]))


4


## Recap

- First and last are adjacent on a circle.
- Solve two lines: drop the last, or drop the first.
- Return the better line total.

Next: [practice](02-circle-practice.ipynb).
